# KA1 Milestone 1: Exploratory audit of the DataCo dataset
Place `DataCoSupplyChainDataset.csv` in `../data/` and run all cells. Nothing here uses private data; the raw file is git-ignored.

In [ ]:
import re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
PATH = Path("../data/DataCoSupplyChainDataset.csv")
if not PATH.exists():
    PATH = Path("data/DataCoSupplyChainDataset.csv")
df = pd.read_csv(PATH, encoding="latin-1")
print(df.shape)

In [ ]:
def find(*keys):
    """Return the first column whose lower-cased name contains all keys."""
    for c in df.columns:
        if all(k in c.lower() for k in keys):
            return c
    return None
TARGET = find("late", "risk")
ORDER_ID = find("order", "id") if find("order", "id") and "item" not in find("order", "id").lower() else None
ORDER_DATE = find("order", "date")
print("target:", TARGET, "| order id:", ORDER_ID, "| order date:", ORDER_DATE)
print(df.dtypes.value_counts())

## 1. Unit of analysis

In [ ]:
print("rows:", len(df))
if ORDER_ID:
    n_orders = df[ORDER_ID].nunique()
    print("distinct orders:", n_orders, "| rows per order (mean):", round(len(df)/n_orders, 2))

## 2. Target and drift in the base rate

In [ ]:
df["_date"] = pd.to_datetime(df[ORDER_DATE], errors="coerce")
df["_q"] = df["_date"].dt.to_period("Q")
print("overall late rate:", round(df[TARGET].mean(), 4))
print("unparseable dates:", int(df["_date"].isna().sum()))
byq = df.groupby("_q")[TARGET].agg(["mean", "size"])
print(byq)
ax = byq["mean"].plot(marker="o", figsize=(7, 3), title="Late-delivery rate by order quarter")
ax.set_ylabel("rate"); plt.tight_layout(); plt.show()

## 3. Missingness

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
print(miss[miss > 0].round(4).to_string())

## 4. Leakage audit
A column is flagged if its name suggests it is recorded after order placement. Review the list by hand; the keyword screen is only a first pass.

In [ ]:
post_keys = ["real", "delivery status", "shipping date", "order status"]
flag = [c for c in df.columns if any(k in c.lower() for k in post_keys) and c != TARGET]
print("post-placement candidates to EXCLUDE:", flag)
tgt_corr = (df.select_dtypes("number").corrwith(df[TARGET]).abs().sort_values(ascending=False).head(10))
print("\nLargest |correlation| with target (inspect for leakage):")
print(tgt_corr.round(3).to_string())

## 5. Redundancy check: scheduled days versus shipping mode

In [ ]:
sched, mode = find("scheduled"), find("shipping", "mode")
if sched and mode:
    print(df.groupby(mode)[sched].nunique())
    print(df.groupby(mode)[TARGET].mean().round(3))

## 6. Order-placement feature contract (draft)
Columns not flagged above and not identifiers or credentials are candidate predictors; finalize after reviewing the audit.

In [ ]:
id_like = [c for c in df.columns if any(k in c.lower() for k in ["email", "password", "fname", "lname", "street", "id", "image", "zipcode"])]
candidates = [c for c in df.columns if c not in flag + id_like + [TARGET, "_date", "_q"]]
print(len(candidates), "candidate columns:")
print(candidates)